# Notebook 01 — Full Data Loading & Data Quality Audit

This notebook is the first stage of the project. It loads the supplied dataset **as-is**, validates the file structure, profiles every column, identifies all data-quality risks, and writes an audit report.

**Non-destructive policy:** the source file is read-only. No rows, characters, encodings, or original columns are overwritten.

## Audit objectives

- Verify encoding, delimiter, header, record count, and field count.
- Detect true missing values and placeholder missing values such as `-`.
- Measure whitespace, Unicode, punctuation, mixed-language, and review-length issues.
- Inspect duplicate rows and duplicate review texts.
- Check rating validity and metadata consistency.
- Produce reproducible CSV/JSON reports for the next preprocessing stage.

In [ ]:
from pathlib import Path
import csv, re, json, hashlib, unicodedata, statistics
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

DATA_PATH = Path('data/Hotel Arabic-Reviews Dataset.txt')
if not DATA_PATH.exists(): DATA_PATH = Path('/home/ubuntu/work_sentiment/sentiment_analysis_submission/data/Hotel Arabic-Reviews Dataset.txt')
REPORT_DIR = Path('reports'); REPORT_DIR.mkdir(exist_ok=True)
print(DATA_PATH.resolve())

In [ ]:
raw = DATA_PATH.read_bytes()
sha256 = hashlib.sha256(raw).hexdigest()
print('bytes:', len(raw)); print('sha256:', sha256)
print('BOM:', raw[:4])
print('decoded preview:', raw[:120].decode('utf-16')[:100])
assert raw[:2] in (b'\xff\xfe', b'\xfe\xff'), 'Expected a UTF-16 BOM.'

In [ ]:
# Structural audit using the original delimiter and encoding.
field_counts=Counter(); logical_rows=0; malformed=[]
with open(DATA_PATH,encoding='utf-16',newline='') as f:
    reader=csv.reader(f,delimiter='\t'); header=next(reader)
    for line_no,row in enumerate(reader,start=2):
        logical_rows += 1; field_counts[len(row)] += 1
        if len(row)!=len(header): malformed.append({'line':line_no,'fields':len(row)})
print('header:',header); print('logical rows:',logical_rows); print('field counts:',dict(field_counts)); print('malformed:',len(malformed))

In [ ]:
df = pd.read_csv(DATA_PATH, sep='\t', encoding='utf-16', dtype=str, keep_default_na=False)
print(df.shape); display(df.head(3))
print('dtypes:',df.dtypes.to_dict())

## 1. Missingness and placeholder values

A dash is not a valid user type, room type, or number of nights. It is a placeholder for unknown/missing information. We count it without changing the source.

In [ ]:
missing_tokens=['','-','--','N/A','NA','n/a','null','None']
missing_report=[]
for col in df.columns:
    s=df[col].astype(str)
    exact={tok:int((s.str.strip()==tok).sum()) for tok in missing_tokens if (s.str.strip()==tok).any()}
    missing_report.append({'column':col,'empty':int((s.str.strip()=='').sum()),'dash':int((s.str.strip()=='-').sum()),'placeholder_total':sum(exact.values()),'tokens':exact})
missing_df=pd.DataFrame(missing_report); display(missing_df)
missing_df.to_csv(REPORT_DIR/'missingness_report.csv',index=False)

## 2. Whitespace, Unicode, and textual noise

These are quality flags, not destructive edits. They show where normalization is needed in Notebook 02.

In [ ]:
def whitespace_stats(s):
    s=s.astype(str)
    return {'leading_or_trailing':int((s!=s.str.strip()).sum()),'repeated_spaces':int(s.str.contains(r'\s{2,}',regex=True).sum()),'embedded_newline':int(s.str.contains(r'[\r\n\t]',regex=True).sum())}
text_cols=['Hotel name','user type','room type','nights','review']
ws=pd.DataFrame([{'column':c,**whitespace_stats(df[c])} for c in text_cols]); display(ws); ws.to_csv(REPORT_DIR/'whitespace_report.csv',index=False)
review=df['review'].astype(str)
unicode_counts=Counter(ch for text in review for ch in text if unicodedata.category(ch).startswith('C'))
print('control characters in reviews:',unicode_counts.most_common(20))
print('review length:',review.str.len().describe(percentiles=[.5,.9,.95,.99]).to_dict())

## 3. Duplicates and label quality

In [ ]:
ratings=pd.to_numeric(df['rating'],errors='coerce')
print('invalid/missing ratings:',int(ratings.isna().sum()),'outside 1..5:',int((~ratings.isna() & ~ratings.between(1,5)).sum()))
print('duplicate full rows:',int(df.duplicated().sum()))
review_counts=review.value_counts()
print('review-text groups repeated:',int((review_counts>1).sum()))
print('extra duplicate occurrences:',int(review_counts[review_counts>1].sub(1).sum()))
print('rating counts:'); display(df['rating'].value_counts(dropna=False).sort_index().to_frame('count'))

## 4. Semantic consistency checks

Some values look like room descriptions but occur in `user type`. We flag them for human review rather than silently moving them between columns.

In [ ]:
room_keywords=r'غرفة|جناح|استوديو|شقة|سرير|توأم|ديلوكس|قياسية|سوبيريور'
user_suspect=df['user type'].astype(str).str.contains(room_keywords,case=False,regex=True,na=False)
print('user_type values matching room-description patterns:',int(user_suspect.sum()))
print(df.loc[user_suspect,['user type','room type']].value_counts().head(20))
print('unique hotels:',df['Hotel name'].nunique(),'unique users:',df['user type'].nunique(),'unique rooms:',df['room type'].nunique(),'unique nights:',df['nights'].nunique())

In [ ]:
# Language/script and punctuation indicators for the review column.
script_report={
 'arabic_chars': int(review.str.count(r'[\u0600-\u06ff]').sum()),
 'latin_chars': int(review.str.count(r'[A-Za-z]').sum()),
 'digits': int(review.str.count(r'\d').sum()),
 'question_exclamation': int(review.str.count(r'[؟?!]').sum()),
 'urls': int(review.str.contains(r'https?://|www\.',regex=True).sum())
}
print(script_report)

In [ ]:
audit_summary={
 'source':str(DATA_PATH.resolve()),'sha256':sha256,'encoding':'UTF-16LE with BOM','delimiter':'TAB',
 'records':int(len(df)),'columns':list(df.columns),'malformed_rows':len(malformed),
 'duplicate_full_rows':int(df.duplicated().sum()),'repeated_review_groups':int((review_counts>1).sum()),
 'extra_duplicate_review_occurrences':int(review_counts[review_counts>1].sub(1).sum()),
 'invalid_or_missing_ratings':int(ratings.isna().sum()),'ratings_outside_1_to_5':int((~ratings.isna() & ~ratings.between(1,5)).sum()),
 'user_type_room_like_values':int(user_suspect.sum()),'missingness':missing_report,'whitespace':ws.to_dict('records'),
 'review_length':{k:float(v) for k,v in review.str.len().describe().to_dict().items()},'script_indicators':script_report}
(REPORT_DIR/'data_quality_audit.json').write_text(json.dumps(audit_summary,ensure_ascii=False,indent=2),encoding='utf-8')
print(json.dumps(audit_summary,ensure_ascii=False,indent=2)[:5000])

In [ ]:
fig,ax=plt.subplots(1,3,figsize=(17,4))
sns.countplot(data=df,x='rating',ax=ax[0],color='#3568a8'); ax[0].set_title('Ratings')
sns.histplot(review.str.len(),bins=60,ax=ax[1],color='#4c956c'); ax[1].set_title('Review length')
sns.countplot(data=df,x='user type',order=df['user type'].value_counts().head(8).index,ax=ax[2],color='#d9822b'); ax[2].tick_params(axis='x',rotation=45); ax[2].set_title('Top user-type values')
plt.tight_layout(); plt.savefig(REPORT_DIR/'data_quality_overview.png',dpi=150); plt.show()

## Audit conclusion

The source is structurally readable and has no malformed logical rows, but it requires a documented derived-data layer for encoding-aware loading, placeholder missingness, whitespace normalization, Arabic normalization, numeric extraction from `nights`, and quality flags. Notebook 02 performs those transformations without overwriting the source.